<a href="https://colab.research.google.com/github/ese-ada-lovelace-2025/irp-wm722/blob/main/Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path
import random
import glob
import torch
import rasterio
import rasterio
from rasterio.windows import Window
import gc
import os


from google.colab import drive
drive.mount('/content/drive')

device = 'cpu'
if torch.cuda.device_count() > 0 and torch.cuda.is_available():
    print("Cuda installed! Running on GPU!")
    device = 'cuda'
else:
    print("No GPU available!")

Mounted at /content/drive
Cuda installed! Running on GPU!


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/IRP/scripts')
from MaskFilm import MaskRCNN as MaskRCNN2
from Deployer import Deployer

In [ ]:
state_dict = torch.load('/content/drive/MyDrive/IRP/models/UKFoldedOutRetrained.pt')
veto_state_dict = torch.load('/content/drive/MyDrive/IRP/models/VetoModelNew.pt')

model_type = MaskRCNN2
dem_path = '/content/drive/MyDrive/IRP/Test/UK/DEMWestWest.tif'
rgb_path = '/content/drive/MyDrive/IRP/Test/UK/WestRGB/composite.tif'
RESOLUTIONS =  [1, 3, 6, 9, 12, 15]

data = {'FarWest1': {'DEM': '/content/drive/MyDrive/IRP/Test/UK/FarWestDEM.tif',
                     'RGB': '/content/drive/MyDrive/IRP/Test/UK/DartmouthRGB/composite.tif'}}

In [ ]:
def run_deployment(dem_path, rgb_path, zone):

  RASTER = dem_path

  overlap = 512  # pixels

  # split large rasters so dont run out of RAM
  with rasterio.open(RASTER) as src:

      mid_x = src.width // 2
      mid_y = src.height // 2

      windows = {
          "top_left": Window(
              0,
              0,
              min(src.width, mid_x + overlap),
              min(src.height, mid_y + overlap),
          ),

          "top_right": Window(
              max(0, mid_x - overlap),
              0,
              src.width - max(0, mid_x - overlap),
              min(src.height, mid_y + overlap),
          ),

          "bottom_left": Window(
              0,
              max(0, mid_y - overlap),
              min(src.width, mid_x + overlap),
              src.height - max(0, mid_y - overlap),
          ),

          "bottom_right": Window(
              max(0, mid_x - overlap),
              max(0, mid_y - overlap),
              src.width - max(0, mid_x - overlap),
              src.height - max(0, mid_y - overlap),
          ),
      }

  for name, window in windows.items():

    if name == 'top_left' or name == 'top_right':
      continue

    temp_path = f"/content/{name}_temp.tif"

    with rasterio.open(RASTER) as src:

        data = src.read(window=window)

        profile = src.profile.copy()
        profile.update(
            width=int(window.width),
            height=int(window.height),
            transform=src.window_transform(window),
        )

        with rasterio.open(temp_path, "w", **profile) as dst:
            dst.write(data)

    del data
    gc.collect()

    print(f"Running {name}...")

    deployer = Deployer(
        dem_path=temp_path,
        rgb_path=rgb_path,
        model_state_dict=state_dict,
        veto_model_dict=veto_state_dict,
        device=device,
        resolutions=RESOLUTIONS,
        native_res = 1,
        veto_threshold=0.85)

    result = deployer.sweep(stride_frac=0.65)

    (predictions, cleaned, rejected, veto_probability_map, out_transform,
            out_crs, support, coverage) = result

    deployer.merge_predictions(cleaned, out_transform, out_crs, output_path=(f"/content/drive/MyDrive/IRP/results/FinalDeployment/{zone}/predictions_area_{name}.shp"))

    print(f"Saved {name}")

    del deployer
    del result

    gc.collect()
    torch.cuda.empty_cache()

    # Remove temporary ~10 GB raster
    if os.path.exists(temp_path):
        os.remove(temp_path)

    print(f"Cleared {name}")

# Running Rectangular DEMs


In [ ]:
for zone, paths in data.items():
  dem = paths['DEM']
  rgb = paths['RGB']
  run_deployment(dem, rgb, zone)

Running bottom_left...
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:00<00:00, 227MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 206MB/s]


Inference at 1 m:   0%|          | 0/11200 [00:00<?, ?tile/s]

...

Completed 12 m inference: 31/75 tiles predicted, 44 skipped.


Inference at 15 m:   0%|          | 0/48 [00:00<?, ?tile/s]

Completed 15 m inference: 18/48 tiles predicted, 30 skipped.


Vetoing predictions:   0%|          | 0/468 [00:00<?, ?obj/s]

108 objects retained, 360 objects rejected, 43 objects could not be assessed.
Saved 98 objects to /content/drive/MyDrive/IRP/results/FinalDeployment/FarWest1/predictions_area_bottom_right.shp
Saved bottom_right
Cleared bottom_right


# Odd Shape DEMs

# Do DEMWestWest

In [ ]:
deployer = Deployer(
        dem_path='/content/drive/MyDrive/IRP/Test/UK/DEMWestWest.tif',
        rgb_path='/content/drive/MyDrive/IRP/Test/UK/WestRGB/composite.tif',
        model_state_dict=state_dict,
        veto_model_dict=veto_state_dict,
        device=device,
        resolutions=[1, 3, 6, 9],
        native_res = 1,
        veto_threshold=0.85
        )

In [ ]:
(predictions, cleaned, rejected, veto_probability_map, out_transform,
            out_crs, support, coverage) = deployer.sweep(stride_frac = 0.65)

In [ ]:
merged = deployer.merge_predictions(cleaned, out_transform, out_crs, "/content/drive/MyDrive/IRP/results/UK65WestWestOddShape.shp")

In [ ]:
from google.colab import runtime
runtime.unassign()